In [3]:
import pandas as pd 
import numpy as np 


In [70]:
movies=pd.read_csv("C:\\Users\\Jathin Prakash\\Downloads\\movies_metadata.csv")

C:\Users\Jathin Prakash\AppData\Local\Temp\ipykernel_24348\1859465304.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies=pd.read_csv("C:\\Users\\Jathin Prakash\\Downloads\\movies_metadata.csv")


In [71]:
ratings=pd.read_csv("C:\\Users\\Jathin Prakash\\Downloads\\ratings_small.csv")

In [72]:
links=pd.read_csv("C:\\Users\\Jathin Prakash\\Downloads\\links.csv")

In [73]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [74]:
movies=movies[['id','original_title']]

In [75]:
movies=movies.rename(columns={'id':'movieId','original_title':'title'})

In [76]:
movies.head()

,movieId,title
0,862,Toy Story
1,8844,Jumanji
2,15602,Grumpier Old Men
3,31357,Waiting to Exhale
4,11862,Father of the Bride Part II


In [77]:
import pandas as pd
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

# 1. Load your uploaded dataset
df = pd.read_csv("C:\\Users\\Jathin Prakash\\Downloads\\ratings_small.csv")

# 2. Define the exact rating scale based on your data (0.5 to 5.0)
# Surprise requires the columns to be exactly in this order: User, Item, Rating
reader = Reader(rating_scale=(0.5, 5.0))
dataset = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)

# 3. Split into 80% training and 20% testing sets
trainset, testset = train_test_split(dataset, test_size=0.2, random_state=42)

# 4. Initialize and fit the SVD model
# We set a random_state for reproducible results
model = SVD(n_factors=100, lr_all=0.005, reg_all=0.02, random_state=42)
print("Training the SVD model...")
model.fit(trainset)

# 5. Predict on the test set and evaluate accuracy
predictions = model.test(testset)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print(f"\nModel Performance on Test Set:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

Training the SVD model...
RMSE: 0.9024
MAE:  0.6954

Model Performance on Test Set:
RMSE: 0.9024
MAE:  0.6954


In [78]:
links['tmdbId']=links['tmdbId'].astype(float)
# Replace all infinities with NaN
links.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows with NaN in a specific column
links.dropna(subset=['tmdbId'], inplace=True)
links['tmdbId']=links['tmdbId'].astype(int)
links['tmdbId']=links['tmdbId'].astype(str)

In [79]:
movies['movieId']=movies['movieId'].astype(str)

In [80]:
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862


In [81]:
links=links.rename(columns={'movieId':'id','tmdbId':'movieId'})

In [91]:
links.head()

,id,imdbId,movieId
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862


In [90]:
movies.head()

,movieId,title
0,862,Toy Story
1,8844,Jumanji
2,15602,Grumpier Old Men
3,31357,Waiting to Exhale
4,11862,Father of the Bride Part II


In [95]:
k=pd.merge(movies,links,on='movieId',how='inner')
k['title']

0                          Toy Story
1                            Jumanji
2                   Grumpier Old Men
3                  Waiting to Exhale
4        Father of the Bride Part II
                    ...             
45520                        رگ خواب
45521            Siglo ng Pagluluwal
45522                       Betrayal
45523            Satana likuyushchiy
45524                       Queerama
Name: title, Length: 45525, dtype: object

In [92]:
ratings['movieId']=ratings['movieId'].astype(str)

In [102]:
import pandas as pd
import numpy as np
from surprise import SVD, Dataset, Reader

# ==========================================
# 1. LOAD & CLEAN YOUR DATASETS (Fixes the Mismatch Bug)
# ==========================================
# Load files

# CRITICAL FIX: Use 'left' join so missing links do not delete movie titles
metadata_df = pd.merge(movies, links, on='movieId', how='left')


# ==========================================
# 2. TRAIN THE SURPRISE SVD MODEL
# ==========================================
reader = Reader(rating_scale=(0.5, 5.0))
dataset = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# Train on the complete dataset
trainset = dataset.build_full_trainset()
model = SVD(n_factors=100, random_state=42)
model.fit(trainset)


# ==========================================
# 3. ROBUST RECOMMENDATION FUNCTION
# ==========================================
def get_detailed_recommendations(model, ratings_df, metadata_df, target_user_id, n=5):
    # Get all unique integer movie IDs
    all_movies = ratings_df['movieId'].unique()
    
    # Identify movies this specific user has already seen
    rated_movies = ratings_df[ratings_df['userId'] == target_user_id]['movieId'].tolist()
    
    # Filter for unrated movies
    unrated_movies = [movie for movie in all_movies if movie not in rated_movies]
    
    # Generate SVD predictions
    predictions = [model.predict(target_user_id, movie_id) for movie_id in unrated_movies]
    
    # Sort predictions highest to lowest
    predictions.sort(key=lambda x: x.est, reverse=True)
    top_n_predictions = predictions[:n]
    
    detailed_recs = []
    for pred in top_n_predictions:
        m_id = str(pred.iid)  # Ensure item ID is strictly evaluated as integer
        est_rating = pred.est
        
        # Look up the movie metadata
        movie_info = metadata_df[metadata_df['movieId'] == m_id]
        
        if not movie_info.empty:
            row = movie_info.iloc[0]
            title = row['title']
            
        else:
            title = f"Unknown Title (Movie ID: {m_id})"
            
        detailed_recs.append({
            "Movie ID": m_id,
            "Title": title,
            "Predicted Rating": round(est_rating, 2),
        })
        
    return pd.DataFrame(detailed_recs)

# ==========================================
# 4. GENERATE AND PRINT RECOMMENDATIONS
# ==========================================
target_user = 10
recommendations_table = get_detailed_recommendations(model, ratings, metadata_df, target_user_id=target_user, n=5)

print(f"\n--- Top 5 Movie Recommendations for User {target_user} ---")
print(recommendations_table.to_string(index=False))


--- Top 5 Movie Recommendations for User 10 ---
Movie ID                              Title  Predicted Rating
     296 Terminator 3: Rise of the Machines              4.74
    1228     Unknown Title (Movie ID: 1228)              4.64
     858               Sleepless in Seattle              4.63
    1252                      Lonely Hearts              4.61
     905             Die Büchse der Pandora              4.59


In [103]:
from surprise.model_selection import GridSearchCV

# 1. Define a search grid of common optimal parameters
param_grid = {
    'n_factors': [50, 100, 150],  # Latent features capturing user/movie behavior
    'lr_all': [0.002, 0.005, 0.01], # Step size for gradient descent
    'reg_all': [0.02, 0.05, 0.1]   # Penalty term to prevent overfitting
}

print("Searching for optimal hyperparameters...")
# 3-Fold Cross-Validation splits data into 3 parts to test combinations robustly
# n_jobs=-1 utilizes all your computer's CPU cores for parallel speed
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(dataset)

# Print out your optimal numbers
print(f"Best CV RMSE achieved: {gs.best_score['rmse']:.4f}")
print("Best parameters found:", gs.best_params['rmse'])

# 2. Extract and train the absolute best model on the full data
optimized_model = gs.best_estimator['rmse']
optimized_model.fit(trainset)

Searching for optimal hyperparameters...
Best CV RMSE achieved: 0.8889
Best parameters found: {'n_factors': 150, 'lr_all': 0.01, 'reg_all': 0.1}


In [104]:
# now to run on entire ratings

In [105]:
ratings_large=pd.read_csv("C:\\Users\\Jathin Prakash\\Downloads\\ratings.csv")

In [106]:
ratings['userId'].nunique()

671

In [117]:
import pandas as pd
import numpy as np
from surprise import SVD, Dataset, Reader
from surprise.model_selection import GridSearchCV

# =====================================================================
# 1. DATA PREPARATION & CLEANING PIPELINE
# =====================================================================
def load_and_clean_data(ratings_path="C:\\Users\\Jathin Prakash\\Downloads\\ratings_small.csv", movies_path="C:\\Users\\Jathin Prakash\\Downloads\\movies_metadata.csv", links_path="C:\\Users\\Jathin Prakash\\Downloads\\links.csv"):
    """
    Loads ratings and metadata files, handles data mismatches, and merges them.
    """
    ratings_df = pd.read_csv(ratings_path)
    movies_df = pd.read_csv(movies_path) 
    links_df = pd.read_csv(links_path)
    links_df=links_df.rename(columns={'movieId':'id','tmdbId':'movieId'})
    movies_df=movies_df[['id','original_title']]
    movies_df=movies_df.rename(columns={'id':'movieId','original_title':'title'})
    
    # Coerce movie IDs to uniform integers to prevent datatype mismatch bugs
    for df in [ratings_df, movies_df, links_df]:
        df['movieId'] = pd.to_numeric(df['movieId'], errors='coerce')

    ratings_df = ratings_df.dropna(subset=['movieId']).astype({'movieId': 'int64'})
    movies_df = movies_df.dropna(subset=['movieId']).astype({'movieId': 'int64'})
    links_df = links_df.dropna(subset=['movieId']).astype({'movieId': 'int64'})

    # Use left join so titles are preserved even if link IDs are missing
    metadata_df = pd.merge(movies_df, links_df, on='movieId', how='left')
    
    return ratings_df, metadata_df


# =====================================================================
# 2. MODEL TUNING AND TRAINING PIPELINE
# =====================================================================
def optimize_and_train_model(ratings_df):
    """
    Runs Grid Search Cross-Validation to find the best SVD hyperparameters, 
    and returns both the trained optimized model and the full trainset.
    """
    # Convert pandas dataframe into a Surprise dataset format
    reader = Reader(rating_scale=(0.5, 5.0))
    dataset = Dataset.load_from_df(ratings_df[['userId', 'movieId', 'rating']], reader)
    trainset = dataset.build_full_trainset()

    print("Searching for optimal SVD hyperparameters via Grid Search...")
    param_grid = {
        'n_factors': [50, 100], 
        'lr_all': [0.005], 
        'reg_all': [0.02, 0.05]
    }
    
    # Perform GridSearch across 3 splits using all available CPU cores
    gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
    gs.fit(dataset)

    print(f"Best Cross-Validation RMSE: {gs.best_score['rmse']:.4f}")
    print(f"Best Parameters: {gs.best_params['rmse']}")
    
    # Extract the optimal estimator and train it on the full training matrix
    model = gs.best_estimator['rmse']
    print("Training the final model on the complete dataset...")
    model.fit(trainset)
    print("Model training complete.\n")
    
    return model, trainset


# =====================================================================
# 3. HIGH-SPEED VECTORIZED RECOMMENDATION FUNCTION
# =====================================================================
def get_optimized_recommendations(model, trainset, metadata_df, ratings_df, target_user_id, n=5):
    """
    Generates high-speed movie recommendations for a specific user using 
    compiled engine matrix testing. Includes an integrated cold-start fallback.
    """
    # Cold Start Handler: Check if user exists in the system
    try:
        inner_user_id = trainset.to_inner_uid(target_user_id)
    except ValueError:
        print(f"User {target_user_id} not found in historical data. Falling back to popular choices.")
        if metadata_df is not None and not metadata_df.empty:
            return metadata_df.head(n)
        return pd.DataFrame({"Movie ID": ratings_df['movieId'].value_counts().index[:n]})
        
    # Vectorized step: Bulk pull all user-item pairs where user hasn't given a rating
    unrated_pairs = trainset.build_anti_testset(fill=None)
    user_unrated_pairs = [pair for pair in unrated_pairs if pair[0] == target_user_id]
    
    # Bulk predict ratings inside the compiled internal architecture
    predictions = model.test(user_unrated_pairs)
    
    # Sort from highest rating score to lowest and extract top N slice
    predictions.sort(key=lambda x: x.est, reverse=True)
    top_n_predictions = predictions[:n]
    
    # Merge titles and map clickable IMDb/TMDb links
    detailed_recs = []
    for pred in top_n_predictions:
        m_id = int(pred.iid)
        est_rating = pred.est
        
        movie_info = metadata_df[metadata_df['movieId'] == m_id]
        if not movie_info.empty:
            row = movie_info.iloc[0]
            title = row['title']
            imdb_link = f"https://www.imdb.com/title/tt{str(int(row['imdbId'])).zfill(7)}/" if pd.notna(row.get('imdbId')) else "N/A"
            tmdb_link = f"https://www.themoviedb.org/movie/{int(row['tmdbId'])}" if pd.notna(row.get('tmdbId')) else "N/A"
        else:
            continue
            
        detailed_recs.append({
            "Movie ID": m_id,
            "Title": title,
            "Predicted Rating": round(est_rating, 2),
            "IMDb Link": imdb_link,
        })
        
    return pd.DataFrame(detailed_recs)


# =====================================================================
# 4. EXECUTION FLOW
# =====================================================================

# Step A: Load and process files
ratings_data, movie_metadata = load_and_clean_data(
    ratings_path="C:\\Users\\Jathin Prakash\\Downloads\\ratings_small.csv", 
    movies_path="C:\\Users\\Jathin Prakash\\Downloads\\movies_metadata.csv", 
    links_path="C:\\Users\\Jathin Prakash\\Downloads\\links.csv"
)

# Step B: Optimize hyperparameters and fit the model globally
optimized_model, full_trainset = optimize_and_train_model(ratings_data)

# Step C: Call recommendation engine whenever needed
target_user = 1
recommendations_table = get_optimized_recommendations(
    model=optimized_model, 
    trainset=full_trainset, 
    metadata_df=movie_metadata, 
    ratings_df=ratings_data, 
    target_user_id=target_user, 
    n=5
)

print(f"--- Optimized Recommendations for User {target_user} ---")
print(recommendations_table.to_string(index=False))

C:\Users\Jathin Prakash\AppData\Local\Temp\ipykernel_24348\1731908555.py:14: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_df = pd.read_csv(movies_path)


Searching for optimal SVD hyperparameters via Grid Search...
Best Cross-Validation RMSE: 0.8962
Best Parameters: {'n_factors': 50, 'lr_all': 0.005, 'reg_all': 0.05}
Training the final model on the complete dataset...
Model training complete.

--- Optimized Recommendations for User 1 ---
 Movie ID                    Title  Predicted Rating                             IMDb Link
      858     Sleepless in Seattle              3.61 https://www.imdb.com/title/tt0108160/
      913  The Thomas Crown Affair              3.55 https://www.imdb.com/title/tt0155267/
      926             Galaxy Quest              3.54 https://www.imdb.com/title/tt0177789/
      318 The Million Dollar Hotel              3.54 https://www.imdb.com/title/tt0120753/


In [118]:
target_user = 10
recommendations_table = get_optimized_recommendations(
    model=optimized_model, 
    trainset=full_trainset, 
    metadata_df=movie_metadata, 
    ratings_df=ratings_data, 
    target_user_id=target_user, 
    n=5
)

print(f"--- Optimized Recommendations for User {target_user} ---")
print(recommendations_table.to_string(index=False))

--- Optimized Recommendations for User 10 ---
 Movie ID                Title  Predicted Rating                             IMDb Link
      858 Sleepless in Seattle              4.51 https://www.imdb.com/title/tt0108160/
      899      Broken Blossoms              4.49 https://www.imdb.com/title/tt0009968/
      926         Galaxy Quest              4.46 https://www.imdb.com/title/tt0177789/


In [119]:
import pickle

# ... (Assuming you already ran your data cleaning and model optimization from earlier) ...

# Bundle the essential components together
artifacts_to_save = {
    'model': optimized_model,
    'trainset': full_trainset,
    'metadata_df': movie_metadata,
    'ratings_df': ratings_data
}

# Save everything to a single file
with open('standalone_svd_recommendation_system.pkl', 'wb') as file:
    pickle.dump(artifacts_to_save, file)

print("Successfully stored the model and data artifacts!")

Successfully stored the model and data artifacts!


In [3]:
pip install nltk


   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   --------------------------- ------------ 1.0/1.6 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 6.9 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [regex]
   ------------- -------------------------- 1/3 [click]
   ------------- -------------------------- 1/3 [click]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   --


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import nltk
nltk.download('vader_lexicon')

from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Initialize the analyzer
sia = SentimentIntensityAnalyzer()


[nltk_data] Downloading package vader_lexicon to C:\Users\Jathin
[nltk_data]     Prakash\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [5]:
df=pd.read_csv("C:\\Users\\Jathin Prakash\\Desktop\\final_dataset.csv")

In [6]:
#genre part


def create_user_genre_profile(target_user_id, ratings_df, metadata_df, genre_matrix, movie_id_to_idx):
    """
    Creates a 'taste profile' vector for a specific user.
    """
    # 1. Get all movies the user rated 4 or 5 stars
    user_ratings = ratings_df[ratings_df['userId'] == target_user_id]
    highly_rated = user_ratings[user_ratings['rating'] >= 4.0]['movieId'].tolist()
    
    # 2. Find the index of these movies in your genre matrix
    # We only care about movies that exist in our genre_matrix
    indices = [movie_id_to_idx[m_id] for m_id in highly_rated if m_id in movie_id_to_idx]
    
    # 3. Create the profile by averaging the genre vectors of their favorite movies
    if indices:
        # This sums the vectors and divides by the count, resulting in 
        # a vector where high values represent the user's most-watched genres.
        profile = genre_matrix[indices].mean(axis=0)
    else:
        # If no ratings, return a neutral/empty vector
        profile = np.zeros(genre_matrix.shape[1])
        
    return profile
# VADER PART;


# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Calculate sentiment for each tag in your dataset
# We group by movieId to get the average sentiment for that movie
df['vader_score'] = df['tags'].apply(lambda x: sia.polarity_scores(x)['compound'])
vader_map = df.groupby('movieId')['vader_score'].mean().to_dict()


def get_vader_score(m_id, vader_map):
    # Get the score from the map, default to 0.0 if the movie has no tags
    raw_score = vader_map.get(m_id, 0.0)
    # Normalize VADER (-1 to 1) to a (0 to 1) range
    return (raw_score + 1) / 2

# Inside your recommendation loop:
# vader_score = get_vader_score(m_id, vader_map)
# final_score = (0.5 * svd_score) + (0.3 * genre_score) + (0.2 * vader_score)

AttributeError: 'float' object has no attribute 'encode'